In [ ]:
from dataclasses import dataclass, replace, asdict, field
from typing import get_type_hints
from typing import List, Dict
from pathlib import Path

import csv
import numpy as np
import pickle
 
from RRAM import (
    CurrentSolver,
    Representate,
    exceptions,
    Percolation,
    Temperature,
    ElectricField,
    Generation
)

In [3]:
def read_csv_to_dic(cvs_path: str) -> List[Dict[str, str]]:
    """
    Reads a CSV file and returns its contents as a list of dictionaries.
    Args:
        cvs_path (str): The name of the CSV file to read.

    Returns:
        list: A list of dictionaries, where each dictionary represents a row in the CSV file.
    """
    with open(cvs_path, mode="r") as archivo:
        reader = csv.DictReader(archivo)
        data = [row for row in reader]
    return data


In [ ]:
@dataclass
class SimulationParameters:
    device_size: float
    atom_size: float
    x_size: int
    y_size: int
    num_trampas: int
    init_simulation_time: float
    total_simulation_time: float
    num_pasos: int
    paso_guardar: int
    voltaje_final_reset: float
    voltaje_final_set: float
    initial_voltaje: float
    initial_current: float
    initial_elec_field: float
    initial_temperatura: float
    T_0: float

    num_max_vacantes: int = field(init=False)
    paso_temporal: float = field(init=False)
    paso_potencial: float = field(init=False)

    def __post_init__(self):
        self.num_max_vacantes = int(0.9 * (self.device_size / self.atom_size) ** 2)
        self.paso_temporal = self.total_simulation_time / self.num_pasos
        self.paso_potencial = self.voltaje_final_reset / self.num_pasos

    @staticmethod
    def from_dict(d: dict):
        # Usar get_type_hints para asegurar que obtienes tipos reales
        field_types = get_type_hints(SimulationParameters)

        mapping = {
            "voltaje_final_reset": "voltaje_final",
            "T_0": "init_temp",
            "initial_temperatura": "init_temp",
        }

        kwargs = {}
        for k in field_types:
            src = mapping.get(k, k)

            # Verificar que la clave existe en el diccionario
            if src not in d:
                raise KeyError(f"La clave '{src}' no existe en el diccionario")

            # Debug: ver qué tipo y valor estamos procesando
            # print(f"Campo: {k}, Tipo: {field_types[k]}, Fuente: {src}, Valor: {d[src]}")

            try:
                kwargs[k] = field_types[k](d[src])
            except Exception as e:
                print(f"Error al convertir {k}: {e}")
                print(
                    f"Tipo esperado: {field_types[k]}, tipo real: {type(field_types[k])}"
                )
                raise

        return SimulationParameters(**kwargs)
    

In [9]:
@dataclass(frozen=True)

class SimulationConstants:
    vibration_frequency: float
    migration_energy: float
    drift_coefficient: float
    cte_red: float
    recom_enchancement_factor: float
    decaimiento_concentracion: float
    activation_energy: float
    gamma: float
    ohm_resistence: float
    pb_metal_insul: float
    permitividad_relativa: float
    I_0: float
    r_termica_percola: float
    r_termica_no_percola: float
    factor_generacion: float
    recombination_energy: float

    @staticmethod
    def from_dict(d: dict):
        # Obtiene los tipos reales de SimulationConstants, no de SimulationParameters
        field_types = get_type_hints(SimulationConstants)

        # Alias si es necesario; si no, mapa vacío sirve
        mapping = {}

        kwargs = {}
        for k in field_types:
            src = mapping.get(k, k)

            if src not in d:
                raise KeyError(f"La clave '{src}' no existe en el diccionario")

            try:
                kwargs[k] = field_types[k](d[src])
            except Exception as e:
                print(f"Error al convertir {k} con valor {d[src]}: {e}")
                raise

        return SimulationConstants(**kwargs)
    
        
    def update_gamma(self, nuevo_valor_gamma: float):
        return replace(self, gamma=nuevo_valor_gamma)


In [ ]:
sim_parmtrs = read_csv_to_dic("Init_data/simulation_parameters.csv")
params = SimulationParameters.from_dict(sim_parmtrs[0])

sim_cte = read_csv_to_dic("Init_data/simulation_constants.csv")
ctes = SimulationConstants.from_dict(sim_cte[0])

SimulationConstants(vibration_frequency=10000000000000.0, migration_energy=0.6, drift_coefficient=10.0, cte_red=2.5e-10, recom_enchancement_factor=3000.0, decaimiento_concentracion=1e-09, activation_energy=1.01, gamma=8.0, ohm_resistence=11.5, pb_metal_insul=0.5, permitividad_relativa=20.0, I_0=0.0018, r_termica_percola=500.0, r_termica_no_percola=10.0, factor_generacion=2.0, recombination_energy=1.04)


In [ ]:
def crear_rutas_simulacion(num_simulation: int, state: str) -> dict:
    """
    Crea las rutas necesarias para guardar resultados de la simulación.

    Args:
        num_simulation (int): Número índice de la simulación.
        state (str): Estado de la simulación, puede ser 'set' o 'reset'.

    Returns:
        dict: Diccionario con las rutas Path para 'simulation', 'figures' y 'set'.

    Side-effects:
        Crea las carpetas en disco si no existen.
    """
    simulation_path = Path.cwd() / "Results" / f"simulation_{num_simulation + 1}"
    figures_path = simulation_path / "Figures"
    data_simulation_path = simulation_path / state

    simulation_path.mkdir(parents=True, exist_ok=True)
    figures_path.mkdir(parents=True, exist_ok=True)
    data_simulation_path.mkdir(parents=True, exist_ok=True)

    return {
        "simulation_path": simulation_path,
        "figures_path": figures_path,
        "data_simulation_path": data_simulation_path,
    }

In [ ]:
def cargar_y_representar_estado(pkl_path: Path , figures_path: Path, voltage: float
) -> np.ndarray:
    """
    Carga el estado de configuración desde archivo pkl y genera una imagen de ese estado.

    Args:
        pkl_path (Path): Ruta del pkl con el estado.
        figures_path (Path): Ruta donde se guardará la imagen generada, debe incluir el nombre del archivo CON extensión.

    Returns:
        np.ndarray: Matriz con el estado inicial cargado.
    """
    with open(f"{pkl_path}.pkl", "rb") as f:
        actual_state = pickle.load(f)

    Representate.RepresentateState(actual_state, voltage, str(figures_path))

    return actual_state

In [ ]:
def crear_matriz_datos(num_pasos: int) -> np.ndarray:
    """
    Crea una matriz de datos inicializada con ceros.
    Esta función genera una matriz de tipo numpy.ndarray con un número de filas
    igual a `num_pasos` y un número de columnas determinado por las cabeceras
    de los datos: "Tiempo simulacion [s]", "Voltaje [V]", "Intensidad [A]".
    La matriz se inicializa con valores de tipo float64.
    Args:
        num_pasos (int): Número de filas que tendrá la matriz, representando
                         los pasos de simulación.
    Returns:
        np.ndarray: Matriz de dimensiones (num_pasos, 3) inicializada con ceros.
    """

    # Defino las cabeceras de los archivos csv
    header_files = "Tiempo simulacion [s],Voltaje [V],Intensidad [A]"
    colunm_number = len(header_files.split(","))
    return np.zeros((num_pasos, colunm_number), dtype=np.float64)


In [ ]:
def procesar_filamentos_creados(
    filamentos,
    CF_creado,
    voltage,
    voltage_CF_creado,
    indice_filamento,
    actual_state,
    num_simulation,
):
    """
    Detecta filamentos nuevos, actualiza su estado y guarda imágenes e
    imágenes correspondientes, además guarda el estado actual en PKL.

    Args:
        filamentos (list): Lista de filamentos en el estado actual.
        CF_creado (np.ndarray): Vector booleano que indica si cada filamento fue creado.
        voltage (float): Voltaje actual.
        voltage_CF_creado (np.ndarray): Array para registrar voltajes de creación.
        indice_filamento (int): Índice actual para la asignación de voltaje.
        actual_state (np.ndarray): Estado actual del sistema.
        num_simulation (int): Número de simulación.

    Returns:
        int: Índice actualizado para el filamento.
    """
    existentes = CurrentSolver.Existe_filamentos(filamentos, len(CF_creado))
    filamentos_nuevos = [i for i, v in enumerate(existentes) if v and not CF_creado[i]]

    for i in filamentos_nuevos:
        CF_creado[i] = True
        voltage_CF_creado[indice_filamento] = voltage
        indice_filamento += 1

        nombre_img = (
            Path.cwd()
            / "Results"
            / "Figures"
            / f"simulation_{num_simulation + 1}"
            / f"Filamento_{i + 1}_creado_set_{num_simulation + 1}.png"
        )
        Representate.RepresentateState(actual_state, round(voltage, 3), str(nombre_img))
        print(f"El filamento {i + 1} se ha creado en el voltaje {voltage}")
        
        # Guardar estado actual en archivo pkl
        nombre_pkl = Path.cwd() / "Results" / f"simulation_{num_simulation + 1}" / "set" / f"filamento_{i + 1}_creado_set_{num_simulation + 1}.pkl"

        with open(nombre_pkl, "wb") as f:
            pickle.dump(actual_state, f)

    return indice_filamento


In [ ]:
def PP_set(
    num_simulation: int,
    params: SimulationParameters,
    sim_ctes: SimulationConstants,
    CF_ranges: List[tuple],
    CF_creado: np.ndarray,
):
    """
    Executes the first part (PP) of the simulation set process for a resistive random-access memory (RRAM) device.
    This function simulates the physical processes involved in the formation of conductive filaments (CFs) 
    in an RRAM device during the set operation. It initializes the simulation environment, updates the 
    state of the system, and records the results at each simulation step.
    Args:
        num_simulation (int): The simulation number (used for naming and saving results).
        params (SimulationParameters): Object containing simulation parameters such as device size, 
            total simulation time, number of steps, and voltage settings.
        sim_ctes (SimulationConstants): Object containing simulation constants such as gamma, 
            factor of generation, and material properties.
        CF_ranges (List[tuple]): List of tuples defining the ranges for conductive filaments.
        CF_creado (np.ndarray): Boolean array indicating whether each conductive filament has been created.
    Raises:
        exceptions.MaxVacantesException: Raised if the maximum number of vacancies is exceeded.
        exceptions.NoPercolationException: Raised if the system does not percolate.
        exceptions.HighPercolationVoltageException: Raised if the percolation voltage is too high.
        exceptions.NullResistanceException: Raised if the resistance becomes null during the simulation.
    Notes:
        - The function creates directories for saving simulation results and figures.
        - It uses various helper functions and classes for tasks such as representing the state, 
          solving currents, and calculating probabilities.
        - The simulation stops early if certain conditions are met, such as exceeding the maximum 
          number of vacancies or reaching the final set voltage.
    Returns:
        None
    """
    
    # Pongo el nombre de la simulación y un salto de línea
    print(f"\nSimulacion {num_simulation + 1}")
    
    # Declaro todas las variables que voy a usar exclusivamente en la primera parte (PP) del set.

    # Rutas de guardado: 
    rutas = crear_rutas_simulacion(num_simulation, state="set")

    # Cargo y represento el estado inicial de configuración
    actual_state = cargar_y_representar_estado(
        Path.cwd() / f"Init_data/initial_state_simulation_{num_simulation}",
        rutas["figures_path"] / f"Initial_state_{num_simulation + 1}.png",
        params.initial_voltaje,
    )
    
    sistema_percola = False

    ocupacion_max_pp_set = 0.35
    factor_vecinos = 1.1  # Factor de aumento de la probabilidad si tiene vecino
    factor_libre = 0.9  # Factor de disminución de la probabilidad si no tiene vecino
    lim_voltage_percolacion = 0.75  # Si el voltaje de percolación es mayor que este valor la simulación no vale la pena seguirla
    temperatura = params.T_0
    current = 0.0
    
    max_vancantes_pp_set = int(ocupacion_max_pp_set * params.num_max_vacantes)

    voltage_CF_creado = np.full(len(CF_ranges), 0.0)
    indice_filamento = 0

    # Inicializo vectores donde almaceno datos
    E_field_vector = np.zeros((actual_state.shape[0]), dtype=np.float64)
    vector_ddp = np.arange(0.000, params.voltaje_final_reset + params.paso_potencial, params.paso_potencial)
    
    # Defino la matriz para almacenar los datos
    data_pp_set = crear_matriz_datos(params.num_pasos)

    config_matrix_pp_set = np.zeros((int((params.num_pasos / params.paso_guardar)), params.x_size, params.y_size))
    

    print("\nComienza la primera parte del set")
    for k in range(0, params.num_pasos):
        # Si se llena el 90 del espacio de la matriz salto a otra simulación. Ponerlo aqui puede dar el problema de que nada mas empezar esté lleno y de error, pero eso NO debe pasar asi q no me preocupa.
        
        total_vacantes = np.sum(actual_state)
        
        if total_vacantes > int(params.num_max_vacantes):
            raise exceptions.MaxVacantesException(k=k-1, voltage=vector_ddp[k-1])
        else:
            # Verifica si el sistema ha percolado
            if not sistema_percola:
                raise exceptions.NoPercolationException()

        # Actualizo el tiempo de simulación y el voltaje
        simulation_time = params.paso_temporal * k
        voltage = vector_ddp[k]

        _, CF_graph = CurrentSolver.Clean_state_matrix(actual_state)

        filamentos = CurrentSolver.Clasificar_CF(CF_graph, params.x_size, params.y_size, CF_ranges)

        # Compruebo si hay filamentos nuevos
        if any(~CF_creado):  # mientras haya alguno sin romper
            filamentos_nuevos = [
                i
                for i, v in enumerate(CurrentSolver.Existe_filamentos(filamentos, len(CF_ranges)))
                if v and not CF_creado[i]
            ]

        for i in filamentos_nuevos:
            CF_creado[i] = True
            voltage_CF_creado[indice_filamento] = voltage
            indice_filamento += 1
            Representate.RepresentateState(
                actual_state,
                round(voltage, 3),
                rutas["simulation_path"] + f"Figures/Filamento_{i + 1}_creado_set_{num_simulation + 1}.png",
            )
            print(f"El filamento {i + 1} se ha creado en el voltaje {voltage}")

        if voltage >= params.voltaje_final_set:
            # Verifica si el sistema ha percolado
            if not sistema_percola:
                raise exceptions.NoPercolationException()

            voltaje_max_set = vector_ddp[k-1]
            tiempo_pp_set = params.paso_temporal * (
                k - 1
            )  # Le quitamos un paso porque se ha superado el voltaje de ruptura

            print(
                "Voltaje final set",
                voltaje_max_set,
                " en el tiempo ",
                tiempo_pp_set,
                "\n",
            )

            # Elimino las filas sobrantes del array de datos y las lleno de nans para eliminarlas luego
            data_pp_set[k - 1 :] = np.nan  # Añadir valores nulos a partir de la fila k
            data_pp_set = data_pp_set[~np.isnan(data_pp_set).any(axis=1)]  # Eliminar filas con valores nulos
            break

        # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
        if Percolation.is_path(actual_state):
            # Si es la primera vez que percola, siste_percola será falso y entra aquí
            if sistema_percola is False:
                voltaje_percolacion = voltage  # Guardo el voltaje de percolación
                print(
                    "\nEl sistema ha percolado en la iteración: ", k,
                    " que corresponde con el voltaje: ",
                    voltaje_percolacion, " con una ocupación del: ",
                    (np.sum(actual_state) / (params.num_max_vacantes)) * 100, " %",
                )
                
                if voltaje_percolacion >= lim_voltage_percolacion:
                    # Si el voltaje de percolación es demasiado alto no va a coincidir con los datos experimentales, y no merece la pena seguir con la simulación
                    raise exceptions.HighPercolationVoltageException(voltage_percola=voltaje_percolacion)
                
                Representate.RepresentateState(
                    actual_state,
                    round(voltaje_percolacion, 3),
                    rutas["figures_path"]
                    + f"Figures/Percola_state_{num_simulation + 1}.png",
                )

                # Cambio la probabilidad de generación de vacantes para controlar la percolación
                sim_ctes.update_gamma(sim_ctes.gamma / sim_ctes.factor_generacion)
                print("Nueva gamma cuando percola el sistema: ", sim_ctes.gamma)

            sistema_percola = True

            exist_cf = CurrentSolver.Existe_filamentos(filamentos, len(CF_ranges))
            cf_clean_matrix = CurrentSolver.Eliminar_filamentos_incompletos(CF_graph, CF_ranges, exist_cf)

            # Si ha percolado uso la corriente de Ohm
            try:
                current, _ = CurrentSolver.OmhCurrent(
                    voltage, cf_clean_matrix, **asdict(sim_ctes)
                )
            except ZeroDivisionError:
                raise exceptions.NullResistanceException(
                    simulation_path=rutas["simulation_path"],
                    figures_path=rutas["figures_path"],
                    voltage=voltage,
                    num_simulation=num_simulation + 1,
                    actual_state=actual_state,
                )

        else:
            sistema_percola = False
            
            mean_field = np.mean(E_field_vector).item()
            # Si no ha percolado uso la corriente de Poole-Frenkel
            current = CurrentSolver.Poole_Frenkel(temperatura, mean_field ,**asdict(sim_ctes)
            ) * (params.device_size)

        # Obtengo los valores del campo eléctrico y la temperatura
        # E_field = ElectricField.SimpleElectricField(voltage, params.device_size)
        temperatura = Temperature.Temperature_Joule(
            voltage, current, sistema_percola, params.T_0, **asdict(sim_ctes)
        )

        # Genero el vector campo eléctrico
        for i in range(0, actual_state.shape[0]):
            E_field_vector[i] = ElectricField.GapElectricField(
                voltage, i, actual_state, **sim_parmtrs[num_simulation]
            )

        # calcular las probabilidades por fila
        prob_generacion_fila = np.minimum(
            [
                Generation.Generate(
                    params.paso_temporal,
                    E_field_vector[i],
                    temperatura,
                    **asdict(sim_ctes),
                )
                for i in range(params.x_size)
            ],
            1,
        )
                            
        # Calculo la probabilidad de generación o recombinación para ello recorro toda la matriz
        for i in range(params.x_size):
            base_prob = prob_generacion_fila[i]
            for j in range(params.y_size):
                if actual_state[i, j] == 0:
                    if total_vacantes < max_vancantes_pp_set:
                        # Compruebo si tiene una vacate cerca
                        if Generation.vecinos_horizontales(actual_state, i, j):
                            prob_generacion = base_prob * factor_vecinos
                        else:
                            prob_generacion = base_prob * factor_libre
                    else:
                        prob_generacion = 0  # LO hago para que no se generen más vacantes y no se llene el sistema

                    random_number = np.random.rand()
                    if random_number < prob_generacion:
                        actual_state[i, j] = 1  # Generación de una vacante


        data_pp_set[k - 1] = np.array([simulation_time, voltage, current])

        # Guardo el estado actual CADA paso_guardar PASOS MONTECARLO
        if k % params.paso_guardar == 0:
            config_matrix_pp_set[int(k / params.paso_guardar) - 1] = actual_state


In [4]:
ruta_raiz = Path.cwd()
    
carpeta_results = ruta_raiz / "Results"

print("carpeta_results", carpeta_results)

carpeta_results c:\Users\Usuario\Documents\GitHub\RRAM_Simulation\Results
